# သင်ခန်းစာ ၁၃ - এဂျင့်မှတ်ဉာဏ်


## Setup

ဤနိုုတ်ဘုတ်မှာ **Microsoft Agent Framework** (MAF) ကို အသုံးပြုပြီး **အမြဲတမ်းမှတ်ဥာဏ်** ရှိသော ခရီးသွားဘူကင်အေဂျင့်တစ်ဦးကို ဘယ်လိုတည်ဆောက်ရမည်ကို ပြသထားသည်။

အမျိုးအစားများကွဲပြားသော အေဂျင့်မှတ်ဥာဏ်များ — အလုပ်လုပ်၊ ချုပ်ငြိမ်း၊ နှင့် ရေရှည် — မည်သို့အေဂျင့်သည် စကားပြောများအတွင်း သတင်းအချက်အလက်များကို သိမ်းဆည်းပြီး အသုံးပြုသည့်နည်းကိုသင်လေ့လာမည်ဖြစ်သည်။

**လိုအပ်ချက်များ:**
- Azure AI Foundry စီမံကိန်းတစ်ခုတွင် chat မော်ဒယ်တစ်ခု (ဥပမာ `gpt-4o-mini`) ကို ထည့်သွင်းထားပြီးဖြစ်ရမည်။
- Azure CLI ဖြင့် ဝင်ရောက်ထားရမည် — သင့် terminal တွင် `az login` ကိုအသုံးပြုပါ။
- `AZURE_AI_PROJECT_ENDPOINT` — သင့် Azure AI Foundry စီမံကိန်း၏ endpoint။
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ထည့်သွင်းထားသည့် မော်ဒယ်အမည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## အေးဂျင့်မှတ်ဉာဏ်အမျိုးအစားများ

AI အေးဂျင့်များသည် တစ်ခုချင်းစီ အထူးပြုရာမှာအသုံးပြုသော မတူညီသောမှတ်ဉာဏ်အမျိုးအစားများကို အကျိုးပြုနိုင်ပါသည်။

### အလုပ်လုပ်မှတ်ဉာဏ်
စကားပြောဆိုမှု အကြောင်းအရာကိုယ်တိုင် — တစ်စက်တစ်ခုအတွင်းထပ်မံပေးပို့သောမက်ဆေ့ချ်များ။ အေးဂျင့်သည် coherence ကို ထိန်းသိမ်းရန် အဲဒီသတင်းပေးပို့မှုလိုင်း အလိုအလျောက် အရင်ပိုင်းမက်ဆေ့ချ်များကို ပြန်ကျော်ကြည့်နိုင်ပါသည်။ MAF တွင် ၎င်းကို **`agent.create_session()`** ဖြင့်ဖန်တီးပြီး `AgentSession` တစ်ခုအား ပြန်လည်ပေးသည်။

### အနည်းကြာမြင့်မှတ်ဉာဏ်
တာဝန် သို့မဟုတ် အစည်းအဝေးအတွင်းသာ သက်တမ်းရှိပြီး အမြဲတမ်း သိမ်းဆည်းထားခြင်း မရှိသော အချက်အလက်များ။ ဥပမာအားဖြင့် အေးဂျင့်သည် မကြာခဏပြောင်းလဲသော စီစဉ်မှုလမ်းညွှန်ဆွေးနွေးမှုအတွင်း အတည်ပြုချက်များစုဆောင်းပြီး နောက်ဆုံးခရီးစဉ် ရလာဒ်ထုတ်လုပ်ရန် အသုံးပြုနိုင်ပါသည်။

### ရုပ်ရှင်ကြာမြင့်မှတ်ဉာဏ်
**အစည်းအဝေးများအတွင်းမှာ ဆက်လက်တည်ရှိနေသော** ကြိုက်နှစ်သက်ထားမှုနှင့် အတည်ပြုချက်များ။ ပြန်လာသောအသုံးပြုသူသည် သူတို့၏အစားအသောက်ကန့်သတ်ချက်များ သို့မဟုတ် ခရီးသွားပုံစံကို ထပ်မံ ပြောဆိုရန် မလိုအပ်သင့်ပါ။ ရုပ်ရှင်ကြာမြင့်မှတ်ဉာဏ်များသည် အမြဲတမ်းသိမ်းဆည်းထားသော အပြင်ဘက်စတိုးဆိုင်— ဒေတာဘေ့စ်၊ ဖိုင် သို့မဟုတ် ဗက်တာအညွှန်းတစ်ခုဖြစ်ပြီး အေးဂျင့်ကိရိယာများမှတဆင့် ကူညီကြားနာပေးသည်။


## Working Memory with Sessions

မှတ်ဉာဏ်၏အလွယ်တကူဆုံးပုံစံမှာ စကားပြောဆက်ဆံမှုအစည်းအဝေးဖြစ်ပါတယ်။ သင့်가 동일한 세션(`agent.create_session()`မှ ဖန်တီးထားသော)ကို ဆက်တိုက် `agent.run()` ခေါ်ကြသည်၊ အဲဒီအစည်းအဝေး၏ အပြည့်အစုံအစဉ်အတိုင်းကို အေးဂျင့်သည် မြင်တွေ့နိုင်ပြီး အစောပိုင်းအသေးစိတ်များကို ပြန်လည်မှတ်မိနိုင်ပါသည်။

ခရီးသွားအေးဂျင့်တစ်ဦး ဖန်တီးကာ working memory ကို ပြသကြရအောင်။


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

အဲဂျင့်သည် ဘတ်ဂျက်ကိုမှန်ကန်စွာ မှတ်မိခဲ့သည်မှာ အဆိုပါမက်ဆေ့ဂျ်နှစ်ခုစလုံးသည် session တစ်ခုတည်းကိုမျှဝေခြင်းကြောင့် ဖြစ်သည်။ ဒါဟာ **အလုပ်လုပ်နေသော မှတ်ဉာဏ်** ဖြစ်ပြီး session ၏သက်တမ်းပိုင်ဆိုင်မှုအကုန်အပြီးမှာပင် ရှိနေသည်။ 

### သစ်သော thread အသစ်ဖြစ်လာရင်ဘာဖြစ်မလဲ?

**သစ်သော** session တစ်ခုဖန်တီးမယ်ဆိုရင် အဲဂျင့်ကို ယခင်စကားပြောစာတမ်းအကြောင်း ရှင်းလင်းမှုမရှိတော့ပါ။


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ရေရှည်မှတ်ဉာဏ်ပုံစံ

အသုံးပြုသူနှစ်သက်မှုများကို **အစကားပြောခန်းများအတွင်းဆက်တိုက်** မှတ်မိနိုင်စေရန်၊ စကားပြောခန်းအတွင်းမှမဟုတ်ဘဲ တည်တံ့သော အမှတ်တရသိုလှောင်ရာ ရှိရပါမည်။ ၄င်းသိုလှောင်ရာသို့ ဝင်ရောက်အသုံးပြုရန်အတွက် အေးဂျင့်သည် **ကိရိယာများ** (tools) ကို အသုံးပြုသည် - သိမ်းဆည်းနိုင့် သတင်းအချက်အလက်များကို အသုံးပြုရန်ခေါ်နိုင်သော ဖန်ချင်များဖြစ်သည်။

အောက်တွင်၊ အလွယ်တကူဇယားမှတ်စုံတစ်ခု (production တွင် database သို့ vector index နည်းဖြင့်ပါပံ့ပိုးနိုင်) ကို အခြေခံ၍ ဖန်တီးကာ၊ အေးဂျင့် အသုံးပြုနိုင်သော ကိရိယာများအဖြစ် ဖော်ပြထားသည်။

### အဆောက်အအုံ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — ပထမဆုံးအသုံးပြုသူ သက်တမ်းပြည့်ခရီးစဉ်မှာ စာနာခြင်း

Sarah သည် ပထမဆုံး အကြိမ် ဧည့်ခံသည်။ ကိုယ်စားလှယ်သည် သူမ၏ ကြိုက်နှစ်သက်မှုများကို ကိရိယာများမှတဆင့် သိုလှောင်ပြီး ဟိုတယ်များ အကြံပေးရန် အသုံးပြုသင့်သည်။


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah returns weeks later

Sarah သည် **အခြားအသစ်သော thread** တစ်ခု (session အသစ်တစ်ခုကို အတုယူခြင်း) ကို စတင်သည်။ အလုပ်မှတ်ဉာဏ်မှာ စွန့်နင်းနေသော်လည်း၊ ရေရှည် စိတ်ကြိုက်သိမ်းဆည်းထားသော မျှော်မှန်းချက်နေရာတွင် မိမိအချက်အလက်များ နောက်ထပ်ရှိနေဆဲဖြစ်သည်။ Agent သည် အချက်အလက်များကို ရယူ၍ အကြံပြုချက်များကို ကိုယ်ပိုင်အတိုင်းပြုလုပ်သင့်သည်။


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## အနှစ်ချုပ်

ဒီသင်ခန်းစာမှာ သင်သည် agent မှတ်ဉာဏ်အမျိုးအစားသုံးမျိုးနှင့် Microsoft Agent Framework ဖြင့် ထည့်သွင်းဆောင်ရွက်နည်းကို လေ့လာခဲ့သည်။

| မှတ်ဉာဏ် အမျိုးအစား | MAF အစိတ်အပိုင်း | အသက်ရှည်ချိန် |
|---|---|---|
| **အလုပ်လုပ်တဲ့** | `agent.create_session()` | တစ်ခါသော စကားပြောဆိုမှု |
| **တိုတောင်းသော** | စဉ်ဆက်တကြောင်းအတွင်း စုပေါင်းထားသော အကြောင်းအရာ | တစ်လုပ်ငန်း/အစည်းအဝေး |
| **ရှည်လျားသော** | `@tool` function များမှတဆင့် ဝင်ရောက်ရယူသည့် ပြင်ပ ဒေတာသိုလှောင်မှု | အစည်းအဝေး အနှစ်ကြီးတစ်လျှောက် |

### အဓိကသိထားရမယ့်အချက်များ
1. **`agent.create_session()`** သည် အလုပ်လုပ်တဲ့ မှတ်ဉာဏ် ပေးသည် — agent သည် session အတွင်း စကားပြောဆိုမှု အပြည့်အစုံကို မြင်နိုင်သည်။
2. **အကြောင်းအရာမဲ့သော session အသစ်များ** — ရှည်လျားသော မှတ်ဉာဏ် မရှိပါက agent သည် အရင်က စကားများကို မမှတ်မိနိုင်။
3. **`@tool` function များသည် ချိတ်ဆက်မှုကို ဆောင်ရြက်သည်** — agent ကို ဒေတာသိုလှောင်မှုမှ သတင်းအချက်အလက် စာရင်းသိမ်းရန်နှင့် ရယူရန် ခွင့်ပြုသည်။
4. **ပုဂ္ဂိုလိကပြုချက်ကောင်းမွန်ခြင်းသည် အချိန်ကြာလာလျှင် ပိုမိုကောင်းမွန်သည်** — ပိုမိုသိမ်းဆည်းထားသည့် အကြိုက်များမှအရ agent ၏ အကြံပြုချက်များ ပိုမိုကောင်းမွန်လာသည်။

### အမှန်တကယ်အသုံးချနိုင်မှုများ
- **ဖောက်သည်ဝန်ဆောင်မှု**: ဖောက်သည်ရဲ့ သမိုင်းကြောင်းနှင့် အကြိုက်များကို မှတ်ထားရန်
- **ပုဂ္ဂိုလိက အကူအညီပေးသူများ**: ရက်တစ်နေ့၊ ပတ်တစ်ပတ်ဖြတ်သော အကြောင်းအရာ ထိန်းသိမ်းရန်
- **ကျန်းမာရေး**: လူနာအချက်အလက်နှင့် အကြိုက်များကို စောင့်ကြပ်ရန်
- **အီလက်ထရွနစ် ကုန်ရောင်းဝယ်မှု**: သမိုင်းကြောင်းပေါ်တွင် အခြေခံပြီး ပုဂ္ဂိုလိကပြု ဝယ်ယူမှုလုပ်ငန်း

### နောက်တစ်ဆင့်များ
- In-memory dict ကို ဒေတာဘေ့စ် သို့မဟုတ် ဗက်တာစတိုး (ဥပမာ Azure AI Search) ဖြင့်အစားထိုးရန်
- အချိန်ပြည့်လိုအပ်သော သတင်းအချက်အလက်များအတွက် မှတ်ဉာဏ် သက်တမ်းကုန်ဆုံးမှု ထည့်သွင်းရန်
- အတူပူးပေါင်းသော multi-agent စနစ်များ တည်ဆောက်ရန်
- Cognee notebook ကို အသုံးပြု၍ သိပ္ပံအခြေခံ သတင်းအချက်အလက်ပေါ်ပြု မှတ်ဉာဏ် ကိုလေ့လာရန်


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**အသိပေးချက်**  
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့်ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှုအတွက် အပြင်းအထန် ကြိုးပမ်းရင်းဖြစ်သော်လည်း၊ စက်ရုပ်ဘာသာပြန်မှုများတွင် မှားယွင်းမှု သို့မဟုတ် အတိအကျမရှိမှုများ ဖြစ်ပေါ်နိုင်ပါသည်။ မူလစာတမ်းကို မိမိဘာသာစကားဖြင့် ရှိသည့် နောက်ခံစာတမ်းအဖြစ် အတည်ပြုရမည့်အရင်းအမြစ်အဖြစ် လက်ခံသင့်ပါသည်။ အရေးကြီးသော သတင်းအချက်အလက်များအတွက် လူ့ဘာသာပြန်ဝန်ထမ်းမှ တိကျမှန်ကန်သော ဘာသာပြန်ချက်ကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာနိုင်သည့် နားမလည်မှုများ သို့မဟုတ် အဓိပ္ပာယ်မှားယွင်းမှုများနှင့် ပတ်သက်၍ ကျွန်ုပ်တို့သည် တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
